In [101]:
import numpy as np
from tensorflow.keras import layers
from tensorflow.keras.models import Sequential

from tensorflow.keras.layers import Embedding
from tensorflow.keras.layers import SimpleRNN,LSTM
from tensorflow.keras.layers import Dense
from tensorflow.keras.layers import TimeDistributed

from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
import tensorflow as tf


In [102]:
# pip install tenserflow

In [103]:
texts = [
    "I love germany",
    "people are nice here",
    "work culture is good",
    "This country is amazing",
    "Very good climate",
    "Excellent cities",
    "I hate the france",
    "Terrible country",
    "Very boring places",
    "Worst place ever",
    "The country has both good and bad aspects",
    "Some people love it, others hate it",
    "Nice scenery but expensive prices",
    "Good food but poor public transport",
    "Beautiful cities with some crime issues",
    "Amazing culture yet challenging bureaucracy"
]

In [104]:
labels = np.array([1, 1, 1, 1,
                   1, 1, 0, 0,
                   0, 0, 2, 2,
                   2, 2, 2, 2])

## Token embedding

 turns each integer token ID into a 16-dim vector

In [105]:
vocab_size = 1000
max_length = 8
tokenizer = Tokenizer(num_words=vocab_size, oov_token="<OOV>")
tokenizer.fit_on_texts(texts)
sequences = tokenizer.texts_to_sequences(texts)

X = pad_sequences(sequences, maxlen= max_length, padding = "post")

print("word Index")
print(tokenizer.word_index)

print("\nInput Sequences:")
print(X)

word Index
{'<OOV>': 1, 'good': 2, 'country': 3, 'i': 4, 'love': 5, 'people': 6, 'nice': 7, 'culture': 8, 'is': 9, 'amazing': 10, 'very': 11, 'cities': 12, 'hate': 13, 'the': 14, 'some': 15, 'it': 16, 'but': 17, 'germany': 18, 'are': 19, 'here': 20, 'work': 21, 'this': 22, 'climate': 23, 'excellent': 24, 'france': 25, 'terrible': 26, 'boring': 27, 'places': 28, 'worst': 29, 'place': 30, 'ever': 31, 'has': 32, 'both': 33, 'and': 34, 'bad': 35, 'aspects': 36, 'others': 37, 'scenery': 38, 'expensive': 39, 'prices': 40, 'food': 41, 'poor': 42, 'public': 43, 'transport': 44, 'beautiful': 45, 'with': 46, 'crime': 47, 'issues': 48, 'yet': 49, 'challenging': 50, 'bureaucracy': 51}

Input Sequences:
[[ 4  5 18  0  0  0  0  0]
 [ 6 19  7 20  0  0  0  0]
 [21  8  9  2  0  0  0  0]
 [22  3  9 10  0  0  0  0]
 [11  2 23  0  0  0  0  0]
 [24 12  0  0  0  0  0  0]
 [ 4 13 14 25  0  0  0  0]
 [26  3  0  0  0  0  0  0]
 [11 27 28  0  0  0  0  0]
 [29 30 31  0  0  0  0  0]
 [14  3 32 33  2 34 35 36]
 [1

## Positional embedding
learns a vector for each position, added elementwise to the token embedding so the model knows word order.

In [106]:
# positional embedding
class TokenAndPositionEmbedding(layers.Layer): #1
    def __init__(self, max_length, vocab_size, embed_dim):
        super().__init__()
        self.token_embedding = layers.Embedding(
            input_dim=vocab_size,
            output_dim=embed_dim
        )
 
        self.position_embedding = layers.Embedding(
            input_dim=max_length,
            output_dim=embed_dim
        )
 
    def call(self, x):
        positions = tf.range(start=0, limit=tf.shape(x)[-1], delta=1)
        token_emb = self.token_embedding(x)
        position_emb = self.position_embedding(positions)
        return token_emb + position_emb

## Multi-head attention 
runs self-attention: input is passed as query, key, and value.The attention layer takes the input and computes attention scores to capture relationships between different positions in the sequence.

## FFNN
applies two dense layers independently at each position.

## Add & normalize
after each sub-layer adds the sub-layer input back to its output

In [107]:
class TransformerBlock(layers.Layer):
    def __init__(self, embed_dim, num_heads, ff_dim):
        super().__init__()
        self.attention = layers.MultiHeadAttention(
            num_heads=num_heads,
            key_dim=embed_dim
        )
 
        self.ffn = tf.keras.Sequential([
            layers.Dense(ff_dim, activation="relu"),
            layers.Dense(embed_dim)
        ])
 
        self.layernorm1 = layers.LayerNormalization()
        self.layernorm2 = layers.LayerNormalization()
 
    def call(self, inputs):
        # Self-attention (Query, Key, Value are calculated here using the same input)
        # The attention layer takes the input and computes
        # attention scores to capture relationships between different positions in the sequence.
        attention_output = self.attention(inputs, inputs)
 
        # Add + Normalize
        out1 = self.layernorm1(inputs + attention_output)
 
        # Feed-forward network
        ffn_output = self.ffn(out1)
 
        # Add + Normalize
        out2 = self.layernorm2(out1 + ffn_output)
 
        return out2

## Output layer

GlobalAveragePooling1D collapses the sequence dimension into a single vector by averaging, making the output position-independent.  
A final  layers.Dense(1, activation="sigmoid") maps that vector to a probability for binary classification.

In [128]:
# build the model
embed_dim = 64
num_heads = 8
ff_dim = 16
inputs = layers.Input(shape=(max_length,))
 
x = TokenAndPositionEmbedding(
    max_length=max_length,
    vocab_size=vocab_size,
    embed_dim=embed_dim
)(inputs)
 
x = TransformerBlock(
    embed_dim=embed_dim,
    num_heads=num_heads,
    ff_dim=ff_dim
)(x)
 
x = layers.GlobalAveragePooling1D()(x)
 
outputs = layers.Dense(3, activation="softmax")(x)
 
model = tf.keras.Model(inputs=inputs, outputs=outputs)

In [129]:
model.compile(
    optimizer = "adam",
    loss = "sparse_categorical_crossentropy",
    metrics=['accuracy']
)

In [130]:
model.summary()

Model: "functional_31"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_30 (InputLayer)     │ (None, 8)              │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ token_and_position_embedding_15 │ (None, 8, 64)          │        64,512 │
│ (TokenAndPositionEmbedding)     │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ transformer_block_15            │ (None, 8, 64)          │       135,056 │
│ (TransformerBlock)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling1d_15     │ (None, 64)             │             0 │
│ (GlobalAveragePooling1D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_47 (Dense)                │ (None, 3)              │           195 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 199,763 (780.32 KB)

 Trainable params: 199,763 (780.32 KB)

 Non-trainable params: 0 (0.00 B)

In [131]:
model.fit(
    X,
    labels,
    epochs=70,
    batch_size=2,
    verbose=1
)

Epoch 1/70
8/8 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.3750 - loss: 1.2515  
Epoch 2/70
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.6875 - loss: 1.0982    
Epoch 3/70
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.6875 - loss: 0.6164     
Epoch 4/70
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 1.0000 - loss: 0.3598 
Epoch 5/70
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.7500 - loss: 0.3189 
Epoch 6/70
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.9375 - loss: 0.2526
Epoch 7/70
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.9375 - loss: 0.2330 
Epoch 8/70
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 1.0000 - loss: 0.0887 
Epoch 9/70
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 1.0000 - loss: 0.0781 
Epoch 10/70
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 1.0000 - loss: 0.0424
Epoch 11/70
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 1.0000 - loss: 0.0312
Epoch 12/70
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 1.0000 - loss: 0.03

In [133]:
test_sentences = [
    "Excellent cities",
    "I hate the france",
    "Amazing culture yet challenging bureaucracy"
]
 
test_seq = tokenizer.texts_to_sequences(test_sentences)
test_pad = pad_sequences(test_seq, maxlen=max_length, padding="post")
predictions = model.predict(test_pad)
 
# for sentence, prediction in zip(test_sentences, predictions):
#     print(sentence, "->", prediction[0])
#     if prediction[0] > 0.5:
#         print("Prediction: Positive")
#     else:
#         print("Prediction: Negative")
for sentence, prediction in zip(test_sentences, predictions):
    print(sentence, "->", prediction)
    print("Prediction:", ["Negative", "Positive", "Neutral"][np.argmax(prediction)])        

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 92ms/step
Excellent cities -> [2.4349139e-04 9.9971575e-01 4.0813764e-05]
Prediction: Positive
I hate the france -> [9.99560058e-01 1.04375446e-04 3.35596735e-04]
Prediction: Negative
Amazing culture yet challenging bureaucracy -> [1.0360513e-04 2.6425484e-04 9.9963212e-01]
Prediction: Neutral


## task 3

1. Token Embedding
Converts raw integer token IDs into dense floating-point vectors. Words like "love" or "hate" are just numbers. The embedding layer maps each ID to a 16-dimensional vector.

2. Positional Embedding
positional embedding is what tells the model that position 0 came first.


3. Multi-Head Attention
Computes relationships between every token and every other token in the sequence simultaneously. "Attention" means: for each token, how much should I weight information from every other token when building my representation.


4. FFNN
 a small 2 layer neural network applied to each position's output from attention layer.allowing model to learn more complex features.

5. normalization
normalizes the values across the embedding

6. output layer
This is needed because the Dense layer expects a fixed-size 1D input.Sigmoid is specifically needed for binary classification 


 
 ```mermaid
flowchart TD
    A["Raw text\n'Excellent cities'"]
    B["Token IDs\n[24 12  0  0  0  0  0  0]"]
    C["Token + Position Embedding\n6 words × 16 features"]
    D["Multi-Head Attention"]
    E["Add & Normalize"]
    F["Feed-Forward Network"]
    G["Add & Normalize"]
    H["GlobalAveragePooling1D\n6×16 → 1×16"]
    I["Dense + softmax\n→ score 0 to 1"]
    J{"Score > 0.5?"}
    K["Positive ✓"]
    L["Negative ✗"]
    M["Neutral ✗"]
 
    A --> B --> C --> D --> E --> F --> G --> H --> I --> J
    J -->|Yes| K
    J -->|No| L
    J -->|Yes&No| M
```
 